# ปฏิบัติการที่ 2 เอเจนต์แบบตอบสนองอย่างง่าย

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NKSR22/Introduction_to_AI_Labs/blob/main/lab02-intelligent-agents.ipynb)

ปฏิบัติการประกอบหนังสือ *ปัญญาประดิษฐ์เบื้องต้น* (นครินทร์ ศรีปัญญา) บทที่ 1 หัวข้อ 1.5 เอเจนต์อัจฉริยะ

## วัตถุประสงค์

เมื่อทำปฏิบัติการนี้จบ ผู้เรียนสามารถ

1. เขียนโปรแกรมเอเจนต์แบบตอบสนองอย่างง่าย (Simple Reflex Agent) สำหรับโลกเครื่องดูดฝุ่นตามรหัสเทียมในหนังสือได้
2. วัดผลเอเจนต์ด้วยเกณฑ์วัดสมรรถนะ (Performance Measure) หลายแบบ และอธิบายได้ว่าทำไมความมีเหตุผลขึ้นกับเกณฑ์ที่เลือก
3. เพิ่มสถานะภายใน (Internal State) ให้เอเจนต์ เพื่อเปลี่ยนเป็นเอเจนต์แบบตอบสนองที่มีแบบจำลอง (Model-based Reflex Agent) แล้วเปรียบเทียบผลได้
4. จำลองเทอร์โมสตัตแบบเกณฑ์เดียวเทียบกับแบบมีช่วงฮิสเทอรีซิส และอธิบายผลต่อจำนวนครั้งที่คอมเพรสเซอร์ถูกสั่งเปิดปิดได้

## เนื้อหาในหนังสือที่เกี่ยวข้อง

- 1.5.1 เอเจนต์ เซนเซอร์ และตัวกระทำ — ตัวอย่างที่ 1.16 และ 1.17
- 1.5.2 เอเจนต์ที่มีเหตุผลและ PEAS — ตัวอย่างที่ 1.18
- 1.5.4 ประเภทของเอเจนต์ — รหัสเทียม SIMPLE-REFLEX-AGENT, MODEL-BASED-REFLEX-AGENT และตัวอย่างที่ 1.22

## ความรู้พื้นฐานที่ควรมี

- Python เบื้องต้น ได้แก่ ตัวแปร `if` `for` ฟังก์ชัน และคลาสอย่างง่าย (ทบทวนได้ในปฏิบัติการที่ 1 และภาคผนวก ก)
- อ่านหัวข้อ 1.5 ของหนังสือมาก่อน

## วิธีใช้ notebook นี้

- กดปุ่ม **Open in Colab** ด้านบนเพื่อเปิดใน Google Colab แล้วรันทีละเซลล์จากบนลงล่างด้วย `Shift + Enter`
- ถ้ารันเซลล์ข้ามลำดับแล้วเกิดข้อผิดพลาด ให้เลือกเมนู **Runtime → Run all** เพื่อรันใหม่ทั้งหมด
- เซลล์ที่มี `# TODO` ในส่วน "แบบฝึกปฏิบัติ" คือส่วนที่ผู้เรียนต้องเติมโค้ดเอง

| ส่วน | หัวข้อ | ตัวอย่างในหนังสือ |
|:---:|---|---|
| 1 | โลกเครื่องดูดฝุ่นและเอเจนต์แบบตอบสนองอย่างง่าย | 1.16 |
| 2 | เกณฑ์วัดสมรรถนะ | 1.18 |
| 3 | ทำไมเอเจนต์จริงไม่ใช้ตาราง | 1.17 |
| 4 | เอเจนต์ดูดฝุ่นแบบมีแบบจำลอง | 1.18 (เกณฑ์ที่ 2) |
| 5 | เทอร์โมสตัต: เกณฑ์เดียวกับฮิสเทอรีซิส | 1.22 |
| 6 | แบบฝึกปฏิบัติ | — |

## เตรียมเครื่องมือ

ปฏิบัติการนี้ใช้เฉพาะไลบรารีที่มีใน Google Colab อยู่แล้ว ได้แก่ `numpy` (คำนวณเชิงตัวเลข) `pandas` (แสดงผลเป็นตาราง) และ `matplotlib` (วาดกราฟ) จึงไม่ต้องติดตั้งเพิ่ม

> **หมายเหตุ** ป้ายกำกับบนกราฟใช้ภาษาอังกฤษ เพราะ Colab ไม่มีฟอนต์ภาษาไทยติดตั้งไว้ ถ้าใช้ภาษาไทยบนกราฟจะแสดงเป็นสี่เหลี่ยม

In [ ]:
# ไม่ต้องติดตั้งไลบรารีเพิ่ม (ถ้ารันบนเครื่องตนเองแล้วขาดไลบรารี ให้รัน: !pip install -q numpy pandas matplotlib)
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42  # กำหนด seed ให้ผลการสุ่มเหมือนเดิมทุกครั้งที่รัน
plt.rcParams["figure.dpi"] = 100

print("numpy", np.__version__, "| pandas", pd.__version__)

## ส่วนที่ 1 โลกเครื่องดูดฝุ่นและเอเจนต์แบบตอบสนองอย่างง่าย

### 1.1 สภาพแวดล้อม

โลกเครื่องดูดฝุ่น (Vacuum World) ในตัวอย่างที่ 1.16 มีห้อง 2 ห้อง คือ A (ซ้าย) และ B (ขวา) และหุ่นยนต์ดูดฝุ่น 1 ตัว

- **ข้อมูลการรับรู้ (Percept):** คู่ `(ตำแหน่ง, สถานะห้องที่อยู่)` เช่น `("A", "Dirty")` หมายถึง [A, สกปรก] หุ่นยนต์ **มองไม่เห็นอีกห้องหนึ่ง**
- **การกระทำ (Action):** ไปขวา (`Right`) ไปซ้าย (`Left`) ดูดฝุ่น (`Suck`) และไม่ทำอะไร (`NoOp`)
- **สมมติฐาน (ตามตัวอย่างที่ 1.18):** ดูดฝุ่นแล้วห้องสะอาดทันที ฝุ่นไม่กลับมาอีก และถ้าอยู่ห้อง B แล้วสั่งไปขวาจะยังอยู่ห้อง B (ชนผนัง)

ในโค้ดใช้ชื่อภาษาอังกฤษ และแปลเป็นภาษาไทยเมื่อแสดงผลเป็นตาราง

In [ ]:
# ชื่อการกระทำและสถานะของห้อง
RIGHT, LEFT, SUCK, NOOP = "Right", "Left", "Suck", "NoOp"
CLEAN, DIRTY = "Clean", "Dirty"

# คำแปลภาษาไทยสำหรับแสดงผล (ให้ตรงกับตารางในหนังสือ)
THAI = {RIGHT: "ไปขวา", LEFT: "ไปซ้าย", SUCK: "ดูดฝุ่น", NOOP: "ไม่ทำอะไร",
        CLEAN: "สะอาด", DIRTY: "สกปรก", None: "ไม่รู้"}


def percept_to_thai(percept):
    """แปลงข้อมูลการรับรู้ ("A", "Dirty") เป็นข้อความ [A, สกปรก]"""
    location, status = percept
    return f"[{location}, {THAI[status]}]"


class VacuumWorld:
    """สภาพแวดล้อมโลกเครื่องดูดฝุ่น 2 ห้อง (A อยู่ซ้าย B อยู่ขวา)"""

    def __init__(self, status_a=DIRTY, status_b=DIRTY, agent_location="A"):
        self.status = {"A": status_a, "B": status_b}
        self.agent_location = agent_location

    def percept(self):
        """เซนเซอร์: บอกเฉพาะตำแหน่งและสถานะของห้องที่หุ่นยนต์อยู่"""
        return (self.agent_location, self.status[self.agent_location])

    def execute(self, action):
        """ตัวกระทำ: เปลี่ยนสภาพแวดล้อมตามการกระทำ"""
        if action == SUCK:
            self.status[self.agent_location] = CLEAN
        elif action == RIGHT:
            self.agent_location = "B"
        elif action == LEFT:
            self.agent_location = "A"
        # NOOP: ไม่มีอะไรเปลี่ยน

    def exogenous_change(self):
        """การเปลี่ยนแปลงที่ไม่ได้เกิดจากเอเจนต์ (โลกนี้ฝุ่นไม่กลับมา จึงไม่มีอะไรเกิดขึ้น)"""
        pass

    def count_clean_rooms(self):
        return sum(status == CLEAN for status in self.status.values())

    def state_text(self):
        """สถานะของทั้งสองห้องในรูป (A, B) เช่น (สะอาด, สกปรก)"""
        return f"({THAI[self.status['A']]}, {THAI[self.status['B']]})"


world = VacuumWorld()
print("ข้อมูลการรับรู้เริ่มต้น:", percept_to_thai(world.percept()))
print("สถานะจริงของโลก (A, B):", world.state_text())

### 1.2 เอเจนต์แบบตอบสนองอย่างง่าย

หัวข้อ 1.5.4 ให้รหัสเทียมของเอเจนต์ดูดฝุ่นแบบตอบสนองอย่างง่ายไว้ดังนี้

```text
ฟังก์ชัน REFLEX-VACUUM-AGENT([location, status]) คืนค่า action
    ถ้า status = สกปรก      แล้ว คืนค่า ดูดฝุ่น
    มิฉะนั้น ถ้า location = A  แล้ว คืนค่า ไปขวา
    มิฉะนั้น ถ้า location = B  แล้ว คืนค่า ไปซ้าย
```

เอเจนต์นี้ใช้ **ข้อมูลการรับรู้ปัจจุบันเท่านั้น** ไม่มีตัวแปรใดเก็บข้อมูลข้ามขั้นเวลา จึงเขียนเป็นฟังก์ชันธรรมดาที่รับ percept แล้วคืนค่า action ได้ทันที

In [ ]:
def reflex_vacuum_agent(percept):
    """เอเจนต์แบบตอบสนองอย่างง่าย: ตัดสินใจจากข้อมูลการรับรู้ปัจจุบันด้วยกฎเงื่อนไข-การกระทำ"""
    location, status = percept
    if status == DIRTY:
        return SUCK
    elif location == "A":
        return RIGHT
    elif location == "B":
        return LEFT


# ทดสอบกับข้อมูลการรับรู้ทั้ง 4 แบบ เทียบกับตารางฟังก์ชันเอเจนต์ในตัวอย่างที่ 1.16
all_percepts = [("A", CLEAN), ("A", DIRTY), ("B", CLEAN), ("B", DIRTY)]
pd.DataFrame({
    "ข้อมูลการรับรู้": [percept_to_thai(p) for p in all_percepts],
    "การกระทำ": [THAI[reflex_vacuum_agent(p)] for p in all_percepts],
})

**คำถามชวนคิด**

1. ตารางข้างบนมีเพียง 4 แถว แต่ตารางฟังก์ชันเอเจนต์ในตัวอย่างที่ 1.16 ยาวขึ้นเรื่อย ๆ ตามเวลา ทำไมเอเจนต์แบบตอบสนองอย่างง่ายจึงใช้แค่ 4 กฎได้
2. ถ้าเพิ่มห้อง C ทางขวาของห้อง B กฎชุดนี้ยังใช้ได้หรือไม่ ต้องแก้อย่างไร

## ส่วนที่ 2 รันเอเจนต์และวัดสมรรถนะ

### 2.1 วงรอบรับรู้ → ตัดสินใจ → กระทำ

ฟังก์ชัน `run_vacuum` ด้านล่างจำลองวงรอบของเอเจนต์ในรูปที่ 1.8 ทีละขั้นเวลา คือ (1) สภาพแวดล้อมส่งข้อมูลการรับรู้ (2) เอเจนต์เลือกการกระทำ (3) สภาพแวดล้อมเปลี่ยนตามการกระทำ แล้วบันทึกผลแต่ละขั้นไว้ในตาราง

ฟังก์ชันนี้รับ `agent` เป็นอะไรก็ได้ที่ "เรียกใช้ด้วย percept แล้วคืนค่า action" จึงใช้ได้ทั้งกับฟังก์ชันอย่าง `reflex_vacuum_agent` และกับคลาสเอเจนต์ที่มีสถานะภายในในส่วนที่ 4

In [ ]:
MOVES = {RIGHT, LEFT}


def run_vacuum(agent, world, steps):
    """รันเอเจนต์ในโลกเครื่องดูดฝุ่นตามจำนวนขั้น แล้วคืนตารางบันทึกการทำงาน"""
    records = []
    for step in range(1, steps + 1):
        percept = world.percept()          # 1) รับรู้
        action = agent(percept)            # 2) ตัดสินใจ
        world.execute(action)              # 3) กระทำ
        records.append({
            "step": step,
            "percept": percept_to_thai(percept),
            "action": THAI[action],
            "state_after": world.state_text(),
            "clean_rooms": world.count_clean_rooms(),
            "moved": action in MOVES,
        })
        world.exogenous_change()           # โลกเปลี่ยนเอง (ถ้ามี) ก่อนขั้นถัดไป
    return pd.DataFrame(records)


def show_log_thai(log):
    """แสดงตารางบันทึกด้วยหัวคอลัมน์ภาษาไทยแบบเดียวกับตารางในตัวอย่างที่ 1.18"""
    table = log.copy()
    table["cumulative"] = table["clean_rooms"].cumsum()
    table = table[["step", "percept", "action", "state_after", "clean_rooms", "cumulative"]]
    table.columns = ["ขั้น", "การรับรู้", "การกระทำ", "สถานะหลังกระทำ (A, B)",
                     "คะแนนขั้นนี้", "คะแนนสะสม"]
    return table.style.hide(axis="index")


# เริ่มที่ห้อง A ทั้งสองห้องสกปรก รัน 6 ขั้น (ตรงกับตัวอย่างที่ 1.18)
reflex_log = run_vacuum(reflex_vacuum_agent, VacuumWorld(DIRTY, DIRTY, "A"), steps=6)
show_log_thai(reflex_log)

ตารางนี้ควรตรงกับตารางในตัวอย่างที่ 1.18 ทุกแถว ให้สังเกตขั้นที่ 4–6 ซึ่งทั้งสองห้องสะอาดแล้ว แต่เอเจนต์ยังเดินไปมา

### 2.2 เกณฑ์วัดสมรรถนะ 2 แบบ

- **เกณฑ์ที่ 1:** ให้ 1 คะแนนต่อห้องที่สะอาดในแต่ละขั้นเวลา
- **เกณฑ์ที่ 2:** เหมือนเกณฑ์ที่ 1 แต่หัก 1 คะแนนทุกครั้งที่เคลื่อนที่ (เพราะสิ้นเปลืองแบตเตอรี่)

เกณฑ์วัดสมรรถนะเป็นฟังก์ชันของ **ลำดับสถานะของสภาพแวดล้อม** ที่ผู้ออกแบบกำหนด ไม่ใช่สิ่งที่เอเจนต์กำหนดเอง จึงเขียนแยกจากตัวเอเจนต์

In [ ]:
def score_clean_rooms(log):
    """เกณฑ์ที่ 1: 1 คะแนนต่อห้องที่สะอาดในแต่ละขั้นเวลา"""
    return int(log["clean_rooms"].sum())


def score_with_move_penalty(log, move_cost=1):
    """เกณฑ์ที่ 2: เกณฑ์ที่ 1 หักค่าเคลื่อนที่ครั้งละ move_cost คะแนน"""
    return score_clean_rooms(log) - move_cost * int(log["moved"].sum())


print("เอเจนต์แบบตอบสนองอย่างง่าย 6 ขั้น")
print("  เกณฑ์ที่ 1:", score_clean_rooms(reflex_log), "คะแนน   (หนังสือ: 10)")
print("  จำนวนครั้งที่เคลื่อนที่:", int(reflex_log["moved"].sum()), "ครั้ง")
print("  เกณฑ์ที่ 2:", score_with_move_penalty(reflex_log), "คะแนน   (หนังสือ: 6)")

**คำถามชวนคิด**

1. ภายใต้เกณฑ์ที่ 1 เอเจนต์นี้มีเหตุผล แต่ภายใต้เกณฑ์ที่ 2 กลับไม่มีเหตุผล ทั้งที่พฤติกรรมเหมือนเดิมทุกประการ เพราะอะไร
2. ถ้าใช้ "เกณฑ์ที่ 3: ให้คะแนนตามปริมาณฝุ่นที่ดูดได้" เอเจนต์ที่มีเหตุผลอาจทำอะไรที่เราไม่ต้องการ (ดูตัวอย่างที่ 1.18)

## ส่วนที่ 3 ทำไมเอเจนต์จริงไม่ใช้ตาราง

เอเจนต์แบบขับเคลื่อนด้วยตาราง (Table-Driven Agent) ต้องเก็บการกระทำสำหรับ **ทุกลำดับการรับรู้** ถ้าข้อมูลการรับรู้มี $|\mathcal{P}|$ แบบ และทำงาน $T$ ขั้น ตารางต้องมีจำนวนแถว

$$N = \sum_{t=1}^{T} |\mathcal{P}|^{t} = \frac{|\mathcal{P}|^{T+1} - |\mathcal{P}|}{|\mathcal{P}| - 1}$$

โค้ดด้านล่างคำนวณ $N$ ด้วยสูตรด้านขวา (Python คำนวณจำนวนเต็มขนาดใหญ่ได้แม่นยำทุกหลัก) แล้วเทียบกับตัวอย่างที่ 1.17

In [ ]:
def table_rows(num_percepts, lifetime_steps):
    """จำนวนแถวของตารางเอเจนต์ = ผลบวกของ |P|^t เมื่อ t = 1 ถึง T"""
    p, T = num_percepts, lifetime_steps
    return (p ** (T + 1) - p) // (p - 1)


# กรณีที่ 1 โลกเครื่องดูดฝุ่น: |P| = 2 ห้อง x 2 สถานะ = 4
for T in [1, 3, 10]:
    print(f"โลกเครื่องดูดฝุ่น T = {T:>2} ขั้น: {table_rows(4, T):>12,} แถว")

# กรณีที่ 2 เทอร์โมสตัตดิจิทัล: อ่านได้ 15.0-40.0 °C ละเอียด 0.1 °C
thermostat_percepts = round((40.0 - 15.0) / 0.1) + 1
print(f"\nเทอร์โมสตัต: |P| = {thermostat_percepts} ค่า")
print(f"  ทำงาน 3 วินาที: {table_rows(thermostat_percepts, 3):,} แถว (ประมาณ 1.6 x 10^7)")

rows_one_hour = table_rows(thermostat_percepts, 3600)
digits = math.floor(math.log10(rows_one_hour)) + 1   # นับจำนวนหลักด้วย log10 (ตัวเลขยาวเกินกว่าจะพิมพ์ออกมา)
print(f"  ทำงาน 1 ชั่วโมง: ตัวเลขยาว {digits:,} หลัก "
      f"(จำนวนอะตอมในเอกภพที่สังเกตได้ประมาณ 10^80 ยาวเพียง 81 หลัก)")

ผลลัพธ์ข้างบนยืนยันข้อสรุปในหนังสือว่า **เอเจนต์แบบตารางสร้างไม่ได้ในทางปฏิบัติ** ขณะที่ `reflex_vacuum_agent` ในส่วนที่ 1 ใช้เพียง 3 บรรทัดของกฎ ความท้าทายของ AI คือการเขียนโปรแกรมขนาดเล็กที่ให้พฤติกรรมที่ดีได้

## ส่วนที่ 4 เอเจนต์ดูดฝุ่นแบบตอบสนองที่มีแบบจำลอง

### 4.1 แนวคิด

ปัญหาของเอเจนต์ในส่วนที่ 2 คือเมื่ออยู่ห้อง A มันไม่รู้ว่าห้อง B สะอาดแล้ว เพราะสภาพแวดล้อม **สังเกตได้บางส่วน** วิธีแก้คือให้เอเจนต์จำสิ่งที่เคยเห็นไว้ใน **สถานะภายใน** ตามรหัสเทียมในหัวข้อ 1.5.4

```text
ฟังก์ชัน MODEL-BASED-REFLEX-AGENT(percept) คืนค่า action
    คงอยู่ถาวร: state, transition_model, sensor_model, rules, action
    state  ← UPDATE-STATE(state, action, percept, transition_model, sensor_model)
    rule   ← RULE-MATCH(state, rules)
    action ← rule.ACTION
    คืนค่า action
```

ในโค้ดด้านล่าง

- **สถานะภายใน** คือพจนานุกรม `self.model` ที่เก็บว่าแต่ละห้อง สะอาด/สกปรก/ไม่รู้ (`None`)
- **แบบจำลองการเปลี่ยนสถานะ:** "ถ้าขั้นก่อนสั่งดูดฝุ่น ห้องนั้นจะสะอาด"
- **แบบจำลองเซนเซอร์:** "สถานะที่เซนเซอร์บอกคือสถานะจริงของห้องที่อยู่"
- **กฎเพิ่ม 1 ข้อ:** "ถ้ารู้ว่าทั้งสองห้องสะอาดแล้ว ไม่ต้องทำอะไร (NoOp)"

เอเจนต์ชนิดนี้ต้องจำข้อมูลข้ามขั้นเวลา จึงเขียนเป็นคลาส และใช้เมธอด `__call__` เพื่อให้เรียกใช้ได้เหมือนฟังก์ชัน `agent(percept)`

In [ ]:
class ModelBasedVacuumAgent:
    """เอเจนต์แบบตอบสนองที่มีแบบจำลอง: จำสถานะของทั้งสองห้องไว้ในสถานะภายใน"""

    def __init__(self):
        self.model = {"A": None, "B": None}   # สถานะภายใน (None = ยังไม่รู้)
        self.last_action = None
        self.last_location = None

    def update_state(self, percept):
        """UPDATE-STATE: ปรับสถานะภายในจากการกระทำครั้งก่อนและข้อมูลการรับรู้ใหม่"""
        location, status = percept
        # แบบจำลองการเปลี่ยนสถานะ: ผลของการกระทำครั้งก่อน
        if self.last_action == SUCK:
            self.model[self.last_location] = CLEAN
        # แบบจำลองเซนเซอร์: ห้องที่อยู่มีสถานะตามที่เซนเซอร์บอก
        self.model[location] = status

    def choose_action(self, location, status):
        """RULE-MATCH: กฎเงื่อนไข-การกระทำ (เพิ่มกฎ NoOp จากเอเจนต์เดิม)"""
        if status == DIRTY:
            return SUCK
        if all(room_status == CLEAN for room_status in self.model.values()):
            return NOOP
        return RIGHT if location == "A" else LEFT

    def __call__(self, percept):
        self.update_state(percept)
        location, status = percept
        action = self.choose_action(location, status)
        self.last_action, self.last_location = action, location
        return action


model_log = run_vacuum(ModelBasedVacuumAgent(), VacuumWorld(DIRTY, DIRTY, "A"), steps=6)
show_log_thai(model_log)

In [ ]:
# เปรียบเทียบเอเจนต์ 2 แบบด้วยเกณฑ์ทั้งสอง
pd.DataFrame({
    "เอเจนต์": ["ตอบสนองอย่างง่าย", "ตอบสนองที่มีแบบจำลอง"],
    "เกณฑ์ที่ 1": [score_clean_rooms(reflex_log), score_clean_rooms(model_log)],
    "จำนวนครั้งที่เคลื่อนที่": [int(reflex_log["moved"].sum()), int(model_log["moved"].sum())],
    "เกณฑ์ที่ 2": [score_with_move_penalty(reflex_log), score_with_move_penalty(model_log)],
})

ผลควรได้เกณฑ์ที่ 2 เท่ากับ 6 และ 9 คะแนน ตรงกับตัวอย่างที่ 1.18 ซึ่งระบุว่าเอเจนต์ที่ "รู้ว่าทั้งสองห้องสะอาดแล้วจึงหยุดอยู่กับที่" ตั้งแต่ขั้นที่ 4 เสียคะแนนเพียง 1 คะแนน

### 4.2 ทดสอบกับสถานะเริ่มต้นทุกแบบ

ผลจากกรณีเดียวอาจบังเอิญดี จึงควรทดสอบกับสถานะเริ่มต้นทุกแบบ โลกนี้มี 8 แบบ (ห้อง A 2 สถานะ × ห้อง B 2 สถานะ × ตำแหน่งเริ่มต้น 2 ห้อง) รันแบบละ 10 ขั้น

In [ ]:
def evaluate_all_starts(make_agent, steps=10):
    """รันเอเจนต์กับสถานะเริ่มต้นทั้ง 8 แบบ คืนตารางคะแนนทั้งสองเกณฑ์"""
    rows = []
    for status_a in [CLEAN, DIRTY]:
        for status_b in [CLEAN, DIRTY]:
            for start in ["A", "B"]:
                log = run_vacuum(make_agent(), VacuumWorld(status_a, status_b, start), steps)
                rows.append({
                    "เริ่มที่": start,
                    "A, B เริ่มต้น": f"{THAI[status_a]}, {THAI[status_b]}",
                    "เกณฑ์ที่ 1": score_clean_rooms(log),
                    "เกณฑ์ที่ 2": score_with_move_penalty(log),
                })
    return pd.DataFrame(rows)


# make_agent ต้องสร้างเอเจนต์ใหม่ทุกครั้ง เพื่อไม่ให้ความจำจากรอบก่อนติดมา
reflex_results = evaluate_all_starts(lambda: reflex_vacuum_agent)
model_results = evaluate_all_starts(ModelBasedVacuumAgent)

comparison = reflex_results[["เริ่มที่", "A, B เริ่มต้น"]].copy()
comparison["ตอบสนองอย่างง่าย: เกณฑ์ 2"] = reflex_results["เกณฑ์ที่ 2"]
comparison["มีแบบจำลอง: เกณฑ์ 2"] = model_results["เกณฑ์ที่ 2"]
display(comparison)
print("ค่าเฉลี่ยเกณฑ์ที่ 1: ตอบสนองอย่างง่าย =", reflex_results["เกณฑ์ที่ 1"].mean(),
      "| มีแบบจำลอง =", model_results["เกณฑ์ที่ 1"].mean())
print("ค่าเฉลี่ยเกณฑ์ที่ 2: ตอบสนองอย่างง่าย =", reflex_results["เกณฑ์ที่ 2"].mean(),
      "| มีแบบจำลอง =", model_results["เกณฑ์ที่ 2"].mean())

**คำถามชวนคิด**

1. ภายใต้เกณฑ์ที่ 1 เอเจนต์ทั้งสองได้คะแนนเท่ากันทุกกรณี แต่ภายใต้เกณฑ์ที่ 2 ต่างกัน สถานะภายในช่วยในส่วนใด
2. เอเจนต์แบบมีแบบจำลองสมมติว่า "ฝุ่นไม่กลับมา" ถ้าในบ้านจริงฝุ่นกลับมาได้เรื่อย ๆ เอเจนต์ที่หยุดนิ่งตลอดไปจะมีปัญหาอะไร (แบบฝึกปฏิบัติข้อ 3 จะให้ทดลองกรณีนี้)

## ส่วนที่ 5 เทอร์โมสตัต: เกณฑ์เดียวกับช่วงฮิสเทอรีซิส

### 5.1 เอเจนต์เทอร์โมสตัต 2 แบบ

เครื่องปรับอากาศทำความเย็นด้วย **คอมเพรสเซอร์ (Compressor)** ซึ่งเทอร์โมสตัตสั่งเปิด (ON) หรือปิด (OFF) ตามอุณหภูมิห้องที่วัดได้ ตัวอย่างที่ 1.22 เปรียบเทียบเทอร์โมสตัต 2 แบบ ค่าเป้าหมาย (Setpoint) 26 °C

- **แบบเกณฑ์เดียว (เอเจนต์แบบตอบสนองอย่างง่าย):** ถ้า $T > 26$ °C เปิดคอมเพรสเซอร์ มิฉะนั้นปิด
- **แบบมีช่วงฮิสเทอรีซิส (Hysteresis Band) ±0.5 °C (มีสถานะภายใน):**

```text
ถ้า T > 26.5             แล้ว compressor ← ON
มิฉะนั้น ถ้า T < 25.5      แล้ว compressor ← OFF
มิฉะนั้น (25.5 ≤ T ≤ 26.5) แล้ว คงสถานะเดิมของ compressor
```

กฎข้อสุดท้ายต้อง **จำสถานะเดิม** จึงเขียนเป็นคลาสที่มีตัวแปร `self.compressor` เป็นสถานะภายใน

In [ ]:
SETPOINT = 26.0   # ค่าเป้าหมาย (°C)
BAND = 0.5        # ครึ่งความกว้างของช่วงฮิสเทอรีซิส: 26 ± 0.5 °C
ON, OFF = "ON", "OFF"


def single_threshold_thermostat(measured_temp, setpoint=SETPOINT):
    """เทอร์โมสตัตแบบเกณฑ์เดียว: ดูเฉพาะอุณหภูมิที่วัดได้ขณะนี้ (ไม่มีสถานะภายใน)"""
    return ON if measured_temp > setpoint else OFF


class HysteresisThermostat:
    """เทอร์โมสตัตแบบมีช่วงฮิสเทอรีซิส: จำสถานะคอมเพรสเซอร์ไว้ในสถานะภายใน"""

    def __init__(self, setpoint=SETPOINT, band=BAND, initial_state=OFF):
        self.setpoint = setpoint
        self.band = band
        self.compressor = initial_state   # สถานะภายใน

    def __call__(self, measured_temp):
        if measured_temp > self.setpoint + self.band:
            self.compressor = ON
        elif measured_temp < self.setpoint - self.band:
            self.compressor = OFF
        # มิฉะนั้นอยู่ในช่วง: คงสถานะเดิม (ไม่ต้องเขียนคำสั่งใด)
        return self.compressor


def count_switches(actions, initial_state=OFF):
    """นับจำนวนครั้งที่คำสั่งคอมเพรสเซอร์เปลี่ยนจากขั้นก่อนหน้า"""
    switches, previous = 0, initial_state
    for action in actions:
        if action != previous:
            switches += 1
        previous = action
    return switches

### 5.2 ไล่การทำงานตามตารางในตัวอย่างที่ 1.22

ใช้อุณหภูมิที่วัดได้ 8 ขั้นชุดเดียวกับในหนังสือ เริ่มจากคอมเพรสเซอร์ปิด

In [ ]:
book_readings = [26.8, 25.9, 26.2, 25.8, 26.1, 25.4, 25.9, 26.6]  # อุณหภูมิที่วัดได้ (°C) จากตัวอย่างที่ 1.22

hysteresis_agent = HysteresisThermostat()
rows = []
for step, temp in enumerate(book_readings, start=1):
    state_before = hysteresis_agent.compressor      # สถานะภายในก่อนตัดสินใจ
    rows.append({
        "ขั้น": step,
        "อุณหภูมิที่วัดได้ (°C)": temp,
        "สถานะภายในก่อนตัดสินใจ": state_before,
        "ฮิสเทอรีซิส": hysteresis_agent(temp),
        "เกณฑ์เดียว 26 °C": single_threshold_thermostat(temp),
    })
book_table = pd.DataFrame(rows)
display(book_table.style.hide(axis="index"))

print("จำนวนครั้งที่เปลี่ยนสถานะคอมเพรสเซอร์ (เริ่มจาก OFF)")
print("  แบบฮิสเทอรีซิส:", count_switches(book_table["ฮิสเทอรีซิส"]), "ครั้ง   (หนังสือ: 3)")
print("  แบบเกณฑ์เดียว:", count_switches(book_table["เกณฑ์เดียว 26 °C"]), "ครั้ง   (หนังสือ: 7)")

สังเกตขั้นที่ 2 และขั้นที่ 7 ซึ่งวัดได้ 25.9 °C เท่ากัน แต่เทอร์โมสตัตแบบฮิสเทอรีซิสสั่ง ON ในขั้นที่ 2 และ OFF ในขั้นที่ 7 เพราะสถานะภายในต่างกัน ส่วนแบบเกณฑ์เดียวให้คำตอบเดียวกันเสมอสำหรับข้อมูลการรับรู้เดียวกัน

### 5.3 แบบจำลองความร้อนของห้อง

8 ขั้นเวลายังสั้นเกินไปที่จะเห็นผลระยะยาว ส่วนนี้จึงจำลองห้องปรับอากาศ 2 ชั่วโมง ด้วยแบบจำลองความร้อนอันดับหนึ่ง (First-order Thermal Model) ซึ่งเป็นแบบจำลองอย่างง่ายที่ใช้กันทั่วไปในงานควบคุม

$$T_{k+1} = T_k + \Delta t \left[ \frac{T_{out} - T_k}{\tau} - c \, u_k \right]$$

- $T_k$ คืออุณหภูมิห้องจริงที่ขั้น $k$ และ $T_{out}$ คืออุณหภูมิภายนอก ความร้อนไหลเข้าห้องเร็วขึ้นเมื่อภายนอกร้อนกว่าภายในมาก
- $\tau$ คือค่าคงตัวเวลาทางความร้อน (Thermal Time Constant) ของห้อง ค่ามากหมายถึงห้องเปลี่ยนอุณหภูมิช้า
- $u_k = 1$ เมื่อคอมเพรสเซอร์เปิด และ $u_k = 0$ เมื่อปิด ส่วน $c$ คืออัตราการทำความเย็น (°C/s)
- เซนเซอร์มี **สัญญาณรบกวน (Noise)** แบบสุ่มปกติ ส่วนเบี่ยงเบนมาตรฐาน 0.15 °C เอเจนต์จึงเห็นเพียงค่าที่วัดได้ ไม่ใช่อุณหภูมิจริง (สังเกตได้บางส่วน)

ค่าพารามิเตอร์ทั้งหมดเป็น **ค่าสมมติ** ที่เลือกให้พฤติกรรมใกล้เคียงห้องทั่วไป ไม่ได้มาจากห้องจริงห้องใด และใช้สัญญาณรบกวนชุดเดียวกันกับเทอร์โมสตัตทั้งสองแบบ เพื่อให้เปรียบเทียบได้อย่างยุติธรรม

In [ ]:
# พารามิเตอร์ของแบบจำลอง (ค่าสมมติ)
DT = 10.0                       # ช่วงเวลาสุ่มตัวอย่าง (วินาที) เทอร์โมสตัตตัดสินใจทุก 10 วินาที
SIM_HOURS = 2.0                 # ระยะเวลาจำลอง (ชั่วโมง)
N_STEPS = int(SIM_HOURS * 3600 / DT)
OUTDOOR_TEMP = 33.0             # อุณหภูมิภายนอก (°C)
TAU = 1800.0                    # ค่าคงตัวเวลาทางความร้อน 30 นาที (วินาที)
COOLING_RATE = 13.0 / TAU       # ถ้าเปิดคอมเพรสเซอร์ตลอด ห้องจะเย็นลงเข้าใกล้ 33 - 13 = 20 °C
SENSOR_NOISE_STD = 0.15         # ส่วนเบี่ยงเบนมาตรฐานของสัญญาณรบกวนเซนเซอร์ (°C)
INITIAL_TEMP = 26.0             # อุณหภูมิห้องเริ่มต้น (°C) เริ่มที่ค่าเป้าหมาย


def room_next_temp(temp, compressor_on, dt=DT):
    """คำนวณอุณหภูมิห้องในขั้นถัดไปจากแบบจำลองความร้อนอันดับหนึ่ง"""
    heat_gain = (OUTDOOR_TEMP - temp) / TAU
    cooling = COOLING_RATE * (1.0 if compressor_on else 0.0)
    return temp + dt * (heat_gain - cooling)


def make_sensor_noise(n_steps=N_STEPS, std=SENSOR_NOISE_STD, seed=RANDOM_SEED):
    """สร้างสัญญาณรบกวนของเซนเซอร์ล่วงหน้า เพื่อใช้ชุดเดียวกันกับทุกเอเจนต์"""
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, std, size=n_steps)


def simulate_thermostat(agent, noise, initial_temp=INITIAL_TEMP):
    """จำลองวงรอบ วัด → ตัดสินใจ → กระทำ คืนตารางผลทุกขั้นเวลา"""
    temp = initial_temp
    records = []
    for k, sensor_noise in enumerate(noise):
        measured = temp + sensor_noise        # 1) เซนเซอร์วัดอุณหภูมิ (มีสัญญาณรบกวน)
        command = agent(measured)             # 2) เอเจนต์ตัดสินใจ
        records.append({"time_min": k * DT / 60, "true_temp": temp,
                        "measured_temp": measured, "compressor": command})
        temp = room_next_temp(temp, command == ON)   # 3) ห้องเปลี่ยนตามคำสั่ง
    return pd.DataFrame(records)


noise = make_sensor_noise()
sim_single = simulate_thermostat(single_threshold_thermostat, noise)
sim_hysteresis = simulate_thermostat(HysteresisThermostat(), noise)
print(f"จำลอง {N_STEPS} ขั้น (ขั้นละ {DT:.0f} วินาที รวม {SIM_HOURS:.0f} ชั่วโมง)")
sim_hysteresis.head()

### 5.4 กราฟอุณหภูมิและสถานะคอมเพรสเซอร์

กราฟแถวบนแสดงอุณหภูมิที่วัดได้ (จุดสีเทา) และอุณหภูมิจริง (เส้นสีน้ำเงิน) พร้อมเส้นค่าเป้าหมายและขอบช่วงฮิสเทอรีซิส กราฟแถวล่างแสดงคำสั่งคอมเพรสเซอร์ (1 = ON, 0 = OFF) คอลัมน์ซ้ายคือแบบเกณฑ์เดียว คอลัมน์ขวาคือแบบฮิสเทอรีซิส

In [ ]:
COLORS = {"blue": "#1F4E79", "orange": "#E07B39", "green": "#3A7D44", "gray": "#6B6B6B"}


def plot_thermostat_comparison(results, minutes=None):
    """วาดกราฟอุณหภูมิ (แถวบน) และสถานะคอมเพรสเซอร์ (แถวล่าง) ของเทอร์โมสตัตแต่ละแบบ"""
    fig, axes = plt.subplots(2, len(results), figsize=(6 * len(results), 5.5),
                             sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    for col, (title, sim) in enumerate(results.items()):
        data = sim if minutes is None else sim[sim["time_min"] <= minutes]
        ax_temp, ax_comp = axes[0, col], axes[1, col]
        ax_temp.plot(data["time_min"], data["measured_temp"], ".", ms=2,
                     color=COLORS["gray"], alpha=0.5, label="measured")
        ax_temp.plot(data["time_min"], data["true_temp"], color=COLORS["blue"], lw=1.5, label="true")
        ax_temp.axhline(SETPOINT, color=COLORS["green"], lw=1, label="setpoint 26 °C")
        for edge in (SETPOINT - BAND, SETPOINT + BAND):
            ax_temp.axhline(edge, color=COLORS["orange"], lw=1, ls="--")
        switches = count_switches(data["compressor"])
        ax_temp.set_title(f"{title}: {switches} switches")
        ax_temp.set_ylabel("Temperature (°C)")
        ax_temp.set_ylim(SETPOINT - 1.0, SETPOINT + 1.0)   # ใช้สเกลเดียวกันทุกกราฟเพื่อเทียบได้ตรงไปตรงมา
        ax_temp.legend(loc="upper right", fontsize=8)
        ax_comp.step(data["time_min"], (data["compressor"] == ON).astype(int),
                     where="post", color=COLORS["orange"], lw=1)
        ax_comp.set_yticks([0, 1], ["OFF", "ON"])
        ax_comp.set_xlabel("Time (min)")
    fig.tight_layout()
    plt.show()


results = {"Single threshold": sim_single, "Hysteresis ±0.5 °C": sim_hysteresis}
plot_thermostat_comparison(results)

กราฟเต็ม 2 ชั่วโมงทำให้เส้นสถานะคอมเพรสเซอร์ของแบบเกณฑ์เดียวดูเป็นแถบทึบ เพราะเปิดปิดถี่มาก ลองขยายดูเฉพาะ 20 นาทีแรก

In [ ]:
plot_thermostat_comparison(results, minutes=20)

### 5.5 ตารางเปรียบเทียบสมรรถนะ

เกณฑ์วัดสมรรถนะของเทอร์โมสตัตมีหลายข้อที่ขัดกัน (เทียบกับ PEAS ของระบบจัดการพลังงานในอาคาร ตัวอย่างที่ 1.20) ได้แก่ ความสบาย (อุณหภูมิใกล้ค่าเป้าหมาย) พลังงาน (สัดส่วนเวลาที่คอมเพรสเซอร์ทำงาน) และอายุอุปกรณ์ (จำนวนครั้งที่เปิดปิด)

In [ ]:
def summarize_run(sim, name):
    """สรุปสมรรถนะของการจำลองหนึ่งครั้งเป็นแถวของตาราง"""
    hours = len(sim) * DT / 3600
    switches = count_switches(sim["compressor"])
    error = (sim["true_temp"] - SETPOINT).abs()
    return {
        "เทอร์โมสตัต": name,
        "จำนวนครั้งเปิดปิด": switches,
        "เปิดปิดต่อชั่วโมง": round(switches / hours, 1),
        "คอมเพรสเซอร์ทำงาน (%)": round(100 * (sim["compressor"] == ON).mean(), 1),
        "อุณหภูมิเฉลี่ย (°C)": round(sim["true_temp"].mean(), 2),
        "คลาดจาก 26 °C สูงสุด (°C)": round(error.max(), 2),
        "เวลาอยู่ในช่วง 25.5-26.5 °C (%)": round(100 * (error <= BAND).mean(), 1),
    }


summary = pd.DataFrame([summarize_run(sim, name) for name, sim in results.items()])
summary

**สิ่งที่ควรสังเกต**

- เทอร์โมสตัตทั้งสองแบบใช้เวลาทำงานของคอมเพรสเซอร์ (พลังงาน) และรักษาอุณหภูมิเฉลี่ยได้ใกล้เคียงกัน
- แต่แบบเกณฑ์เดียวสั่งเปิดปิดบ่อยกว่าแบบฮิสเทอรีซิสเกิน 10 เท่า ซึ่งคือ **การตัดต่อถี่ (Short Cycling)** ที่ทำให้คอมเพรสเซอร์สึกหรอเร็ว ผลนี้สอดคล้องกับตัวอย่างที่ 1.22 ที่เห็นความต่างได้แม้มีเพียง 8 ขั้น
- แบบฮิสเทอรีซิสแลกด้วยการปล่อยให้อุณหภูมิแกว่งกว้างกว่า (ดูคอลัมน์คลาดจาก 26 °C สูงสุด) แต่ยังอยู่ในช่วง 25.5–26.5 °C เกือบตลอดเวลา

### 5.6 การตัดต่อถี่เกิดจากสัญญาณรบกวนอย่างเดียวหรือไม่

ลองจำลองซ้ำโดยเปลี่ยนขนาดสัญญาณรบกวนของเซนเซอร์ ตั้งแต่ 0 (เซนเซอร์สมบูรณ์แบบ) ถึง 0.3 °C

In [ ]:
noise_levels = [0.0, 0.05, 0.10, 0.15, 0.30]
noise_rows = []
for std in noise_levels:
    noise_k = make_sensor_noise(std=std)
    sim_s = simulate_thermostat(single_threshold_thermostat, noise_k)
    sim_h = simulate_thermostat(HysteresisThermostat(), noise_k)
    noise_rows.append({"สัญญาณรบกวน (°C)": std,
                       "เกณฑ์เดียว: ครั้งเปิดปิด": count_switches(sim_s["compressor"]),
                       "ฮิสเทอรีซิส: ครั้งเปิดปิด": count_switches(sim_h["compressor"])})
pd.DataFrame(noise_rows)

**คำถามชวนคิด**

1. แม้สัญญาณรบกวนเป็น 0 เทอร์โมสตัตแบบเกณฑ์เดียวก็ยังเปิดปิดถี่ เพราะอะไร (ใบ้: อุณหภูมิลดลงเท่าใดใน 1 ขั้นเวลาเมื่อคอมเพรสเซอร์เปิด และจะต่ำกว่า 26 °C เมื่อใด)
2. ทำไมจำนวนครั้งเปิดปิดของแบบฮิสเทอรีซิสแทบไม่เปลี่ยนตามสัญญาณรบกวน
3. หนังสือหมายเหตุไว้ว่าในทางปฏิบัติ เอเจนต์อาจอ่านสถานะคอมเพรสเซอร์จากหน้าสัมผัสช่วยของคอนแทคเตอร์ได้โดยตรง ถ้าทำเช่นนั้น เทอร์โมสตัตแบบฮิสเทอรีซิสยังนับเป็นเอเจนต์ที่ต้องใช้ข้อมูลมากกว่าค่าอุณหภูมิ ณ ขณะนั้นหรือไม่

## ส่วนที่ 6 แบบฝึกปฏิบัติ

เติมโค้ดในตำแหน่งที่มี `# TODO` แล้วรันเซลล์ตรวจคำตอบที่อยู่ถัดไป ถ้ายังไม่ได้เติมโค้ด เซลล์ตรวจจะแจ้งว่า "ยังไม่ได้ทำ" โดยไม่หยุดการทำงานของ notebook

### ข้อ 1 เทอร์โมสตัตแบบฮิสเทอรีซิสสำหรับเครื่องทำความร้อน

ห้องควบคุมไฟฟ้าในพื้นที่หนาวใช้ฮีตเตอร์รักษาอุณหภูมิที่ 20 °C ช่วงฮิสเทอรีซิส ±0.5 °C กฎกลับด้านกับเครื่องปรับอากาศ คือ

- ถ้า $T < 19.5$ °C เปิดฮีตเตอร์ (ON)
- ถ้า $T > 20.5$ °C ปิดฮีตเตอร์ (OFF)
- มิฉะนั้นคงสถานะเดิม

จงเติมเมธอด `__call__` ของคลาส `HysteresisHeater` ให้ทำงานตามกฎนี้

In [ ]:
class HysteresisHeater:
    """เทอร์โมสตัตฮีตเตอร์แบบมีช่วงฮิสเทอรีซิส"""

    def __init__(self, setpoint=20.0, band=0.5, initial_state=OFF):
        self.setpoint = setpoint
        self.band = band
        self.heater = initial_state   # สถานะภายใน

    def __call__(self, measured_temp):
        # TODO: เขียนกฎ 3 ข้อ แล้วคืนค่า self.heater
        #   ถ้า measured_temp < setpoint - band  -> self.heater = ON
        #   ถ้า measured_temp > setpoint + band  -> self.heater = OFF
        #   มิฉะนั้นคงสถานะเดิม
        return None

In [ ]:
# เซลล์ตรวจคำตอบข้อ 1 (ไม่ต้องแก้)
heater_readings = [19.3, 19.8, 20.3, 20.6, 20.2, 19.6, 19.4, 20.0]
expected_heater = [ON, ON, ON, OFF, OFF, OFF, ON, ON]

heater = HysteresisHeater()
heater_actions = [heater(temp) for temp in heater_readings]
if all(action is None for action in heater_actions):
    print("ข้อ 1: ยังไม่ได้ทำ")
elif heater_actions == expected_heater:
    print("ข้อ 1: ถูกต้อง", heater_actions)
else:
    print("ข้อ 1: ยังไม่ถูก")
    print("  ได้   ", heater_actions)
    print("  ควรได้", expected_heater)

### ข้อ 2 เลือกความกว้างของช่วงฮิสเทอรีซิส

ช่วงฮิสเทอรีซิสกว้างช่วยลดการเปิดปิด แต่ทำให้อุณหภูมิแกว่งมากขึ้น จงเติมลูปให้จำลองเทอร์โมสตัตแบบฮิสเทอรีซิสด้วยค่า `band` แต่ละค่า (ใช้ `noise` ชุดเดิมจากส่วนที่ 5) แล้วเก็บผลจาก `summarize_run` ลงในรายการ `sweep_rows` เซลล์ถัดไปจะวาดกราฟแสดงการแลกเปลี่ยน (Trade-off)

> **ข้อควรระวัง** `summarize_run` คำนวณ "เวลาอยู่ในช่วง" จากค่าคงที่ `BAND = 0.5` เสมอ จึงใช้เปรียบเทียบความสบายด้วยเกณฑ์เดียวกันทุกค่า `band` ได้

In [ ]:
band_values = [0.1, 0.25, 0.5, 1.0, 1.5]
sweep_rows = []

for band in band_values:
    # TODO: 1) สร้างเอเจนต์ HysteresisThermostat(band=band)
    #       2) จำลองด้วย simulate_thermostat(agent, noise)
    #       3) เพิ่มผล summarize_run(sim, f"±{band} °C") ลงใน sweep_rows
    #          และเพิ่มคีย์ "band" ในพจนานุกรมผลด้วย
    pass

pd.DataFrame(sweep_rows)

In [ ]:
# เซลล์วาดกราฟข้อ 2 (ไม่ต้องแก้)
if len(sweep_rows) == 0:
    print("ข้อ 2: ยังไม่ได้ทำ")
else:
    sweep = pd.DataFrame(sweep_rows)
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(sweep["band"], sweep["เปิดปิดต่อชั่วโมง"], "o-", color=COLORS["orange"])
    ax1.set_xlabel("Hysteresis half-band (°C)")
    ax1.set_ylabel("Switches per hour", color=COLORS["orange"])
    ax2 = ax1.twinx()
    ax2.plot(sweep["band"], sweep["คลาดจาก 26 °C สูงสุด (°C)"], "s--", color=COLORS["blue"])
    ax2.set_ylabel("Max |T - 26| (°C)", color=COLORS["blue"])
    ax1.set_title("Trade-off: equipment wear vs comfort")
    fig.tight_layout()
    plt.show()

**คำถามข้อ 2:** ถ้าผู้ผลิตคอมเพรสเซอร์แนะนำว่าไม่ควรเปิดปิดเกิน 8 ครั้งต่อชั่วโมง และผู้ใช้ห้องยอมรับอุณหภูมิคลาดจาก 26 °C ได้ไม่เกิน 1.2 °C ควรเลือก `band` ค่าใด เพราะอะไร

*คำตอบ:* (เขียนคำตอบที่นี่)

### ข้อ 3 เมื่อฝุ่นกลับมาได้: ปรับเอเจนต์แบบมีแบบจำลองให้ "ลืม"

ในบ้านจริง ฝุ่นกลับมาได้เรื่อย ๆ คลาส `DustyVacuumWorld` ด้านล่างจำลองว่าหลังทุกขั้นเวลา แต่ละห้องมีโอกาส 10% ที่จะกลับมาสกปรก (สภาพแวดล้อมแบบพลวัตและเฟ้นสุ่ม) รันเซลล์นี้เพื่อดูผลของเอเจนต์ 2 แบบเดิมก่อน

In [ ]:
class DustyVacuumWorld(VacuumWorld):
    """โลกเครื่องดูดฝุ่นที่ฝุ่นกลับมาได้: หลังแต่ละขั้น ห้องหนึ่ง ๆ สกปรกด้วยความน่าจะเป็น dirt_prob"""

    def __init__(self, dirt_prob=0.1, seed=RANDOM_SEED, **kwargs):
        super().__init__(**kwargs)
        self.dirt_prob = dirt_prob
        self.rng = np.random.default_rng(seed)

    def exogenous_change(self):
        for room in ["A", "B"]:
            if self.rng.random() < self.dirt_prob:
                self.status[room] = DIRTY


def average_dusty_score(make_agent, steps=200, seeds=range(10)):
    """คะแนนเฉลี่ยต่อขั้นเวลา (เกณฑ์ที่ 1 และ 2) จากการรันหลาย seed"""
    score1, score2 = [], []
    for seed in seeds:
        log = run_vacuum(make_agent(), DustyVacuumWorld(seed=seed), steps)
        score1.append(score_clean_rooms(log) / steps)
        score2.append(score_with_move_penalty(log) / steps)
    return round(np.mean(score1), 3), round(np.mean(score2), 3)


dusty_rows = []
for name, make_agent in [("ตอบสนองอย่างง่าย", lambda: reflex_vacuum_agent),
                         ("มีแบบจำลอง (จำตลอดไป)", ModelBasedVacuumAgent)]:
    s1, s2 = average_dusty_score(make_agent)
    dusty_rows.append({"เอเจนต์": name, "เกณฑ์ 1 ต่อขั้น": s1, "เกณฑ์ 2 ต่อขั้น": s2})
pd.DataFrame(dusty_rows)

เอเจนต์แบบมีแบบจำลองเดิมเชื่อว่าห้องที่เคยเห็นว่าสะอาดจะสะอาดตลอดไป จึงหยุดนิ่งอยู่ห้องเดียว ปล่อยให้อีกห้องสกปรกโดยไม่รู้ตัว ส่วนเอเจนต์แบบตอบสนองอย่างง่ายเดินไปมาตลอดจึงเสียคะแนนจากการเคลื่อนที่มาก

**โจทย์:** เติมเมธอด `choose_action` ของคลาส `ForgetfulVacuumAgent` ให้เอเจนต์ "ลืม" ข้อมูลของห้องที่ไม่ได้เห็นนานเกิน `memory_steps` ขั้น กล่าวคือ

1. ถ้าห้องที่อยู่สกปรก → ดูดฝุ่น
2. ถ้าอีกห้องหนึ่งรู้ว่าสะอาด **และ** เพิ่งเห็นมาไม่เกิน `memory_steps` ขั้น → ไม่ทำอะไร (NoOp)
3. มิฉะนั้น → ไปตรวจอีกห้องหนึ่ง

คลาสนี้สืบทอดจาก `ModelBasedVacuumAgent` และเพิ่มสถานะภายใน `self.last_seen` (ขั้นเวลาล่าสุดที่เห็นแต่ละห้อง) กับ `self.time` (ขั้นเวลาปัจจุบัน) ไว้ให้แล้ว เป้าหมายคือได้คะแนนเกณฑ์ที่ 2 สูงกว่าเอเจนต์ทั้งสองแบบเดิม

In [ ]:
class ForgetfulVacuumAgent(ModelBasedVacuumAgent):
    """เอเจนต์แบบมีแบบจำลองที่ลืมข้อมูลห้องที่ไม่ได้เห็นนานเกิน memory_steps ขั้น"""

    def __init__(self, memory_steps=5):
        super().__init__()
        self.memory_steps = memory_steps
        self.time = 0
        self.last_seen = {"A": None, "B": None}   # ขั้นเวลาล่าสุดที่เห็นแต่ละห้อง

    def update_state(self, percept):
        super().update_state(percept)
        self.time += 1
        location, _ = percept
        self.last_seen[location] = self.time

    def choose_action(self, location, status):
        other_room = "B" if location == "A" else "A"
        # TODO: เขียนกฎ 3 ข้อตามโจทย์
        #   ใช้ self.model[other_room], self.last_seen[other_room], self.time และ self.memory_steps
        #   (ระวัง: self.last_seen[other_room] อาจเป็น None ถ้ายังไม่เคยเห็นห้องนั้น)
        return None

In [ ]:
# เซลล์ตรวจคำตอบข้อ 3 (ไม่ต้องแก้)
test_agent = ForgetfulVacuumAgent()
if test_agent(("A", CLEAN)) is None:
    print("ข้อ 3: ยังไม่ได้ทำ")
else:
    rows = list(dusty_rows)
    for memory in [2, 5, 10]:
        s1, s2 = average_dusty_score(lambda: ForgetfulVacuumAgent(memory_steps=memory))
        rows.append({"เอเจนต์": f"ลืมเมื่อเกิน {memory} ขั้น", "เกณฑ์ 1 ต่อขั้น": s1, "เกณฑ์ 2 ต่อขั้น": s2})
    display(pd.DataFrame(rows))
    best_old = max(row["เกณฑ์ 2 ต่อขั้น"] for row in dusty_rows)
    _, forgetful_score = average_dusty_score(ForgetfulVacuumAgent)
    if forgetful_score > best_old:
        print(f"ข้อ 3: ผ่าน — เกณฑ์ 2 ต่อขั้น {forgetful_score} สูงกว่าเอเจนต์เดิมที่ดีที่สุด ({best_old})")
    else:
        print(f"ข้อ 3: ยังไม่ผ่าน — เกณฑ์ 2 ต่อขั้น {forgetful_score} ไม่สูงกว่า {best_old}")

**คำถามข้อ 3:** ค่า `memory_steps` น้อยและมากมีผลต่อคะแนนเกณฑ์ที่ 1 และ 2 ต่างกันอย่างไร ถ้าโอกาสที่ฝุ่นกลับมาเพิ่มจาก 10% เป็น 30% ควรปรับ `memory_steps` ไปทางใด

*คำตอบ:* (เขียนคำตอบที่นี่)

## สรุป

- **เอเจนต์แบบตอบสนองอย่างง่าย** ตัดสินใจจากข้อมูลการรับรู้ปัจจุบันด้วยกฎเงื่อนไข-การกระทำ เขียนเป็นฟังก์ชันสั้น ๆ ได้ ไม่ต้องใช้ตารางขนาดมหาศาลแบบเอเจนต์แบบตาราง (ส่วนที่ 1 และ 3)
- **เกณฑ์วัดสมรรถนะ** เป็นตัวกำหนดว่าพฤติกรรมใดมีเหตุผล เอเจนต์ตัวเดียวกันได้ 10 คะแนนภายใต้เกณฑ์ที่ 1 แต่ไม่มีเหตุผลภายใต้เกณฑ์ที่ 2 ซึ่งหักค่าเคลื่อนที่ (ส่วนที่ 2)
- **สถานะภายใน** ช่วยให้เอเจนต์ทำงานในสภาพแวดล้อมที่สังเกตได้บางส่วนได้ดีขึ้น เอเจนต์ดูดฝุ่นที่จำสถานะของห้องได้ 9 คะแนนภายใต้เกณฑ์ที่ 2 เทียบกับ 6 คะแนน (ส่วนที่ 4)
- **เทอร์โมสตัตแบบฮิสเทอรีซิส** ใช้สถานะภายในเพื่อลดการตัดต่อถี่ ในการจำลอง 2 ชั่วโมง แบบเกณฑ์เดียวเปิดปิดบ่อยกว่าแบบฮิสเทอรีซิสเกิน 10 เท่า โดยใช้พลังงานและได้อุณหภูมิเฉลี่ยใกล้เคียงกัน (ส่วนที่ 5)
- **แบบจำลองต้องสอดคล้องกับโลกจริง** เอเจนต์ที่เชื่อว่าฝุ่นไม่กลับมาทำงานแย่ลงเมื่อโลกเปลี่ยน การกำหนดให้ความจำ "หมดอายุ" เป็นวิธีง่าย ๆ ที่ช่วยได้ (แบบฝึกข้อ 3)

**ก้าวต่อไป:** เอเจนต์ในปฏิบัติการนี้ยังทำตามกฎตายตัว เอเจนต์แบบมีเป้าหมายซึ่งค้นหาลำดับการกระทำเพื่อไปถึงเป้าหมายเป็นเนื้อหาของบทที่ 2 และปฏิบัติการที่ 3

## เอกสารอ้างอิง

- Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. บทที่ 2 (โลกเครื่องดูดฝุ่นและโครงสร้างของเอเจนต์)